# Silver Layer — Payment Methods Data Cleaning & Quality Checks
**GlobalMart | Tredence DE Advanced Training**

| | |
|---|---|
| **Source** | `<your-catalog>.bronze.payment_methods` (ADLS Gen2 via Autoloader) |
| **Target** | `<your-catalog>.silver.payment_methods` |

> Small lookup/reference table — `PaymentMethodID` → `MethodName`. This
> is exactly what `dim_payment_method` needs, and what `silver.payments`
> joins against to turn `PM-001` into `"Credit Card"` for reporting.

### What this notebook does
| Step | Action |
|---|---|
| 1 | Setup |
| 2 | Read Bronze + inspect schema |
| 3 | DQ scan — nulls, uniqueness, `_rescued_data` |
| 4 | Transform + write |
| 5 | Verify |

## Step 1 — Setup

In [0]:
from pyspark.sql.functions import *

CATALOG      = "harsh_kumar01_npmentorskool_onmicrosoft_com"
BRONZE_TABLE = "harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.payment_methods"
SILVER_TABLE = "harsh_kumar01_npmentorskool_onmicrosoft_com.silver.payment_methods"

print(f"Reading from : {BRONZE_TABLE}")
print(f"Writing to   : {SILVER_TABLE}")

Reading from : harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.payment_methods
Writing to   : harsh_kumar01_npmentorskool_onmicrosoft_com.silver.payment_methods


## Step 2 — Read Raw Data from Bronze

In [0]:
bronze_df = spark.table(BRONZE_TABLE)
print(f"Total records in Bronze: {bronze_df.count():,}")
bronze_df.display()

Total records in Bronze: 5


PaymentMethodID,MethodName,_rescued_data,_source_file,_ingested_at
PM-001,Credit Card,null,/mnt/virinchy_gbmart_data/payment_methods/payment_methods_010626.csv,2026-07-10T07:00:34.565Z
PM-002,UPI,null,/mnt/virinchy_gbmart_data/payment_methods/payment_methods_010626.csv,2026-07-10T07:00:34.565Z
PM-003,Debit Card,null,/mnt/virinchy_gbmart_data/payment_methods/payment_methods_010626.csv,2026-07-10T07:00:34.565Z
PM-004,Net Banking,null,/mnt/virinchy_gbmart_data/payment_methods/payment_methods_010626.csv,2026-07-10T07:00:34.565Z
PM-005,Cash-on-Delivery,null,/mnt/virinchy_gbmart_data/payment_methods/payment_methods_010626.csv,2026-07-10T07:00:34.565Z


## Step 3 — DQ Scan
Small reference table — check nulls, `PaymentMethodID` uniqueness, and
the standing `_rescued_data` schema-drift check used on every
Autoloader-sourced table.

In [0]:
bronze_df.select(
    count(when(col("PaymentMethodID").isNull(), 1)).alias("null_payment_method_id"),
    count(when(col("MethodName").isNull(), 1)).alias("null_method_name")
).display()

dupe_count = bronze_df.groupBy("PaymentMethodID").count().filter("count > 1").count()
print(f"Duplicate PaymentMethodID count: {dupe_count}")

rescued_count = bronze_df.filter(col("_rescued_data").isNotNull()).count()
print(f"Rows with non-null _rescued_data: {rescued_count} / {bronze_df.count()}")

null_payment_method_id,null_method_name
0,0


Duplicate PaymentMethodID count: 0
Rows with non-null _rescued_data: 0 / 5


## Step 4 — Transform & Write
No DQ issues expected for a table this small and well-formed — just
rename to snake_case and write.

In [0]:
silver_df = bronze_df \
    .withColumnRenamed("PaymentMethodID", "payment_method_id") \
    .withColumnRenamed("MethodName", "method_name") \
    .withColumn("_silver_updated_at", current_timestamp()) \
    .select("payment_method_id", "method_name", "_source_file", "_silver_updated_at")

silver_df.display()

payment_method_id,method_name,_source_file,_silver_updated_at
PM-001,Credit Card,/mnt/virinchy_gbmart_data/payment_methods/payment_methods_010626.csv,2026-07-10T12:54:38.542886Z
PM-002,UPI,/mnt/virinchy_gbmart_data/payment_methods/payment_methods_010626.csv,2026-07-10T12:54:38.542886Z
PM-003,Debit Card,/mnt/virinchy_gbmart_data/payment_methods/payment_methods_010626.csv,2026-07-10T12:54:38.542886Z
PM-004,Net Banking,/mnt/virinchy_gbmart_data/payment_methods/payment_methods_010626.csv,2026-07-10T12:54:38.542886Z
PM-005,Cash-on-Delivery,/mnt/virinchy_gbmart_data/payment_methods/payment_methods_010626.csv,2026-07-10T12:54:38.542886Z


In [0]:
## spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

print(f"Written to {SILVER_TABLE}: {spark.table(SILVER_TABLE).count():,} rows")

Written to harsh_kumar01_npmentorskool_onmicrosoft_com.silver.payment_methods: 5 rows


## Step 5 — Verify

In [0]:
spark.table(SILVER_TABLE).display()

payment_method_id,method_name,_source_file,_silver_updated_at
PM-001,Credit Card,/mnt/virinchy_gbmart_data/payment_methods/payment_methods_010626.csv,2026-07-10T12:54:39.707781Z
PM-002,UPI,/mnt/virinchy_gbmart_data/payment_methods/payment_methods_010626.csv,2026-07-10T12:54:39.707781Z
PM-003,Debit Card,/mnt/virinchy_gbmart_data/payment_methods/payment_methods_010626.csv,2026-07-10T12:54:39.707781Z
PM-004,Net Banking,/mnt/virinchy_gbmart_data/payment_methods/payment_methods_010626.csv,2026-07-10T12:54:39.707781Z
PM-005,Cash-on-Delivery,/mnt/virinchy_gbmart_data/payment_methods/payment_methods_010626.csv,2026-07-10T12:54:39.707781Z


## Reset (if needed)

In [0]:
# spark.sql(f"DROP TABLE IF EXISTS {SILVER_TABLE}")
# print("Reset complete")